# Build Clause-Level CCS (Rev4 & Rev5) — v2 (smt + gdn + obj, with subtree synthesis)

This notebook rebuilds clause-level JSONL catalogs from OSCAL JSON, including **statement (_smt)**, **guidance (_gdn)**, and **objective (_obj)** parts. It also synthesizes text for structural nodes that have no direct prose by concatenating descendant prose (verbatim).


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path

# ====== EDIT THESE PATHS ======
# Point to ORIGINAL OSCAL JSON files
REV5_OSCAL_JSON = Path("/content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.json")
REV4_OSCAL_JSON = Path("/content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.json")

# Output clause-level JSONL (keep the default filenames our pipeline expects)
REV5_OUT_JSONL = Path("/content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl")
REV4_OUT_JSONL = Path("/content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl")

# (Optional) Gold CSVs for validation coverage
REV5_GOLD_CSV = Path("/content/drive/MyDrive/compliance_data/gold_standard_datasets/nist800_53/nist_sp800-53_rev5_gold-set_100q.csv")
REV4_GOLD_CSV = Path("/content/drive/MyDrive/compliance_data/gold_standard_datasets/nist800_53/nist_sp800-53_rev4_gold-set_36q.csv")

print("REV5_OSCAL_JSON exists:", REV5_OSCAL_JSON.exists(), REV5_OSCAL_JSON)
print("REV4_OSCAL_JSON exists:", REV4_OSCAL_JSON.exists(), REV4_OSCAL_JSON)

REV5_OSCAL_JSON exists: True /content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.json
REV4_OSCAL_JSON exists: True /content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.json


In [3]:
import json
import re
from typing import Any, Dict, Iterable, Iterator, List, Optional, Tuple

# -----------------------------
# OSCAL tree walkers
# -----------------------------
def _iter_groups(groups: List[Dict[str, Any]]) -> Iterator[Dict[str, Any]]:
    for g in groups or []:
        if not isinstance(g, dict):
            continue
        yield g
        sub = g.get("groups")
        if isinstance(sub, list):
            yield from _iter_groups(sub)

def _iter_controls(control: Dict[str, Any]) -> Iterator[Dict[str, Any]]:
    """Yield control and nested enhancement controls."""
    if not isinstance(control, dict):
        return
    yield control
    for c in control.get("controls") or []:
        if isinstance(c, dict):
            yield from _iter_controls(c)

def _iter_catalog_controls(catalog: Dict[str, Any]) -> Iterator[Dict[str, Any]]:
    """Yield all controls (including nested enhancements) in the catalog."""
    groups = catalog.get("groups") or []
    for group in _iter_groups(groups):
        for ctrl in group.get("controls") or []:
            if isinstance(ctrl, dict):
                yield from _iter_controls(ctrl)

def _iter_parts(parts: List[Dict[str, Any]], parent_part_id: Optional[str]) -> Iterator[Tuple[Dict[str, Any], Optional[str]]]:
    """Depth-first traversal over parts, returning (part, parent_part_id)."""
    for p in parts or []:
        if not isinstance(p, dict):
            continue
        yield p, parent_part_id
        kids = p.get("parts")
        if isinstance(kids, list) and kids:
            yield from _iter_parts(kids, parent_part_id=(p.get("id") or parent_part_id))

# -----------------------------
# Part id classification / labels
# -----------------------------
_ALLOWED_KINDS = ("smt", "gdn", "obj")

def _classify_part_id(pid: str) -> Optional[str]:
    pid = (pid or "").strip().lower()
    if "_smt" in pid:
        return "smt"
    if "_gdn" in pid:
        return "gdn"
    if "_obj" in pid:
        return "obj"
    return None

def _get_label(part: Dict[str, Any]) -> str:
    # Prefer props[label], else empty.
    props = part.get("props")
    if isinstance(props, list):
        for pr in props:
            if isinstance(pr, dict) and pr.get("name") == "label":
                v = str(pr.get("value", "")).strip()
                if v:
                    return v
    return ""

def _collect_descendant_prose(part: Dict[str, Any]) -> List[str]:
    """Collect descendant prose strings (verbatim), prefixed with labels when present."""
    out: List[str] = []
    kids = part.get("parts")
    if not isinstance(kids, list):
        return out
    for child, _ in _iter_parts(kids, parent_part_id=part.get("id")):
        prose = child.get("prose")
        if isinstance(prose, str) and prose.strip():
            lab = _get_label(child)
            txt = prose.strip()
            if lab:
                out.append(f"{lab} {txt}")
            else:
                out.append(txt)
    return out

def _build_fulltext(part: Dict[str, Any], kind: str) -> Tuple[str, bool, bool]:
    """Return (full_text, has_prose, used_descendants)."""
    prose = part.get("prose")
    has_prose = isinstance(prose, str) and prose.strip()
    base = prose.strip() if has_prose else ""

    # Always attempt to synthesize for structural nodes (no prose) and statement/objective roots.
    part_name = str(part.get("name","")).strip().lower()
    desc = _collect_descendant_prose(part)
    used_desc = False

    if desc and (not base or (kind in ("smt","obj") and part_name in ("statement","objective","assessment-objective"))):
        used_desc = True
        if base:
            base = (base + " " + " ".join(desc)).strip()
        else:
            base = " ".join(desc).strip()

    return base, bool(has_prose), used_desc

# -----------------------------
# Main builder
# -----------------------------
def build_clause_jsonl_records(oscal_json: Dict[str, Any], *, version_label: str) -> List[Dict[str, Any]]:
    catalog = oscal_json.get("catalog")
    if not isinstance(catalog, dict):
        raise ValueError("Input JSON does not have top-level 'catalog' object.")

    meta = catalog.get("metadata") or {}
    prov = {
        "oscal_catalog_version": str(meta.get("version", "")).strip(),
        "oscal_catalog_title": str(meta.get("title", "")).strip(),
        "version_label": version_label,
    }

    out: List[Dict[str, Any]] = []
    seen: set = set()

    for ctrl in _iter_catalog_controls(catalog):
        ctrl_id = str(ctrl.get("id", "")).strip()
        title = str(ctrl.get("title", "")).strip()
        parts = ctrl.get("parts")
        if not isinstance(parts, list):
            continue

        for part, parent_part_id in _iter_parts(parts, parent_part_id=None):
            pid = str(part.get("id", "")).strip()
            if not pid:
                continue

            kind = _classify_part_id(pid)
            if kind is None:
                continue

            pid_norm = pid.lower()
            if pid_norm in seen:
                continue
            seen.add(pid_norm)

            full_text, has_prose, used_desc = _build_fulltext(part, kind)

            # Skip empty-text nodes (common in some OSCAL rev4 guidance parts). These break retrieval indexing.
            if not isinstance(full_text, str) or not full_text.strip():
                continue

            rec = {
                "id": pid_norm,                         # clause id (used for citations)
                "control_id": ctrl_id.lower(),          # base control/enhancement id
                "title": title,
                "text": full_text,                      # verbatim prose or verbatim concatenation of descendants
                "kind": kind,                           # smt | gdn | obj
                "part_name": str(part.get("name", "")).strip(),
                "label": _get_label(part),
                "parent_part_id": (str(parent_part_id).strip().lower() if parent_part_id else ""),
                "has_prose": bool(has_prose),
                "used_descendants": bool(used_desc),
                "provenance": prov,
            }
            out.append(rec)

    return out

def write_jsonl(records: List[Dict[str, Any]], out_path) -> None:
    from pathlib import Path
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

# -----------------------------
# Gold coverage validation (robust parsing + normalization)
# -----------------------------
def _norm_gold_id(gid: str) -> str:
    gid = (gid or "").strip().lower()
    # Normalize control family separator in some gold sets: "ac_11" -> "ac-11"
    gid = re.sub(r"^([a-z]{2})_", r"\1-", gid)
    return gid

def load_id_set(jsonl_path) -> set:
    s = set()
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                if isinstance(obj, dict) and obj.get("id"):
                    s.add(str(obj["id"]).strip().lower())
            except Exception:
                pass
    return s

def gold_missing(jsonl_ids: set, gold_csv_path, label: str):
    from pathlib import Path
    import pandas as pd
    gold_csv_path = Path(gold_csv_path)
    if not gold_csv_path.exists():
        print(f"[Validate:{label}] Gold CSV not found at {gold_csv_path} (skipping)")
        return
    df = pd.read_csv(gold_csv_path, encoding="latin-1")
    all_gold = []
    for _, row in df.iterrows():
        raw = str(row.get("gold_control_path", "")).replace("\r\n", "\n")
        parts = re.split(r"[;\n]+", raw)
        all_gold.extend([_norm_gold_id(x) for x in parts if x and x.strip()])
    uniq = sorted(set(all_gold))
    missing = [x for x in uniq if x not in jsonl_ids]
    print(f"[Validate:{label}] Gold unique ids: {len(uniq)} | Missing unique ids: {len(missing)}")
    if missing:
        print(f"[Validate:{label}] First 40 missing:")
        for x in missing[:40]:
            print("  -", x)

In [4]:
def build_one(oscal_path: Path, out_jsonl: Path, version_label: str) -> None:
    data = json.load(open(oscal_path, 'r', encoding='utf-8'))
    recs = build_clause_jsonl_records(data, version_label=version_label)
    write_jsonl(recs, out_jsonl)
    print(f"[OK] {version_label}: wrote {len(recs)} clause docs -> {out_jsonl}")

# Build
build_one(REV5_OSCAL_JSON, REV5_OUT_JSONL, "rev5")
build_one(REV4_OSCAL_JSON, REV4_OUT_JSONL, "rev4")

[OK] rev5: wrote 6831 clause docs -> /content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl
[OK] rev4: wrote 6563 clause docs -> /content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl


In [5]:
# Sanity check: confirm expected clause IDs exist in the output JSONL
def sample_ids(jsonl_path: Path, n: int = 50):
    ids = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            obj = json.loads(line)
            ids.append(obj.get('id'))
            if len(ids) >= n:
                break
    return ids

rev5_ids = sample_ids(REV5_OUT_JSONL, 50)
rev4_ids = sample_ids(REV4_OUT_JSONL, 50)

def has_kind(ids, needle):
    return any(needle in (i or '') for i in ids)

print("Rev5 sample IDs:", rev5_ids[:15])
print("Rev5 has _smt:", has_kind(rev5_ids, "_smt"))
print("Rev5 has _gdn:", has_kind(rev5_ids, "_gdn"))
print("Rev5 has _obj:", has_kind(rev5_ids, "_obj"))

print("Rev4 sample IDs:", rev4_ids[:15])
print("Rev4 has _smt:", has_kind(rev4_ids, "_smt"))
print("Rev4 has _gdn:", has_kind(rev4_ids, "_gdn"))
print("Rev4 has _obj:", has_kind(rev4_ids, "_obj"))

Rev5 sample IDs: ['ac-1_smt', 'ac-1_smt.a', 'ac-1_smt.a.1', 'ac-1_smt.a.1.a', 'ac-1_smt.a.1.b', 'ac-1_smt.a.2', 'ac-1_smt.b', 'ac-1_smt.c', 'ac-1_smt.c.1', 'ac-1_smt.c.2', 'ac-1_gdn', 'ac-1_obj', 'ac-1_obj.a', 'ac-1_obj.a-1', 'ac-1_obj.a-2']
Rev5 has _smt: True
Rev5 has _gdn: True
Rev5 has _obj: True
Rev4 sample IDs: ['ac-1_smt', 'ac-1_smt.a', 'ac-1_smt.a.1', 'ac-1_smt.a.2', 'ac-1_smt.b', 'ac-1_smt.b.1', 'ac-1_smt.b.2', 'ac-1_gdn', 'ac-1_obj', 'ac-1.a_obj', 'ac-1.a.1_obj', 'ac-1.a.1_obj.1', 'ac-1.a.1_obj.1.a', 'ac-1.a.1_obj.1.b', 'ac-1.a.1_obj.1.c']
Rev4 has _smt: True
Rev4 has _gdn: True
Rev4 has _obj: True


In [6]:
# Optional validation: coverage against gold_control_path
# (This uses robust parsing: splits on newlines + semicolons, and normalizes "ac_11" -> "ac-11".)

rev5_id_set = load_id_set(REV5_OUT_JSONL)
rev4_id_set = load_id_set(REV4_OUT_JSONL)

gold_missing(rev5_id_set, REV5_GOLD_CSV, "rev5")
gold_missing(rev4_id_set, REV4_GOLD_CSV, "rev4")

[Validate:rev5] Gold unique ids: 336 | Missing unique ids: 0
[Validate:rev4] Gold unique ids: 88 | Missing unique ids: 0
